In [ ]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# # kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import cv2
import pydicom
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/yuvrajbhatkariya/data-2/new_labeled.csv")
print(df.shape)
df.sample(10)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ==========================================
# PATHS
# ==========================================
DATA_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
LABELS_PATH = Path("/kaggle/input/datasets/yuvrajbhatkariya/data-2/new_labeled.csv")

# ==========================================
# TARGETS
# ==========================================
TARGETS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

# ==========================================
# CURRENT MASKS
# ==========================================
MASKS = {
    "Sagittal": [1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1],
    "Coronal":  [0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1],
    "Axial":    [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1]
}

print("Setup complete.")

In [ ]:
# ==========================================
# LOAD DATA
# ==========================================

labels_df = pd.read_csv(LABELS_PATH)
series_df = pd.read_csv(DATA_DIR / "train_series.csv")

print("Labels shape :", labels_df.shape)
print("Series shape :", series_df.shape)

print("\nLabels columns:")
print(labels_df.columns.tolist())

print("\nSeries columns:")
print(series_df.columns.tolist())

In [ ]:
# ==========================================
# MERGE SERIES + LABELS
# ==========================================

merged_df = pd.merge(
    series_df,
    labels_df,
    on="StudyInstanceUID",
    how="inner"
)

print("Merged shape:", merged_df.shape)

print("\nMerged columns:")
print(merged_df.columns.tolist())

In [ ]:
# ==========================================
# CHECK ANATOMICAL PLANES
# ==========================================

print("\n" + "="*60)
print("ANATOMICAL PLANE COUNTS")
print("="*60)

print(
    merged_df["Anatomical_Plane"]
    .value_counts(dropna=False)
)

In [ ]:
# ==========================================
# UNIQUE STUDIES PER PLANE
# ==========================================

unique_studies_by_plane = (
    merged_df
    .groupby("Anatomical_Plane")["StudyInstanceUID"]
    .nunique()
)

print("\n" + "="*60)
print("UNIQUE STUDIES PER PLANE")
print("="*60)

print(unique_studies_by_plane)

In [ ]:
# ==========================================
# POSITIVE LABEL COUNTS BY PLANE
# ==========================================

positive_counts = (
    merged_df
    .groupby("Anatomical_Plane")[TARGETS]
    .sum()
)

print("\n" + "="*60)
print("POSITIVE LABEL COUNTS BY PLANE")
print("="*60)

positive_counts

In [ ]:
# ==========================================
# POSITIVE LABEL PERCENTAGE BY PLANE
# ==========================================

positive_rates = (
    merged_df
    .groupby("Anatomical_Plane")[TARGETS]
    .mean() * 100
)

print("\n" + "="*60)
print("POSITIVE LABEL RATE (%) BY PLANE")
print("="*60)

positive_rates.round(2)

In [ ]:
# ==========================================
# CLEAN PLANE COMPARISON
# ==========================================

plane_comparison = (
    merged_df
    .groupby("Anatomical_Plane")[TARGETS]
    .mean()
    .mul(100)
    .round(2)
    .T
)

print("\n" + "="*60)
print("ABNORMALITY × PLANE")
print("="*60)

print(plane_comparison)

In [ ]:
# ==========================================
# CHECK ONE STUDY ACROSS PLANES
# ==========================================

study_id = merged_df["StudyInstanceUID"].iloc[0]

one_study = merged_df[
    merged_df["StudyInstanceUID"] == study_id
]

print("\n" + "="*60)
print("ONE STUDY ACROSS PLANES")
print("="*60)

print(
    one_study[
        [
            "StudyInstanceUID",
            "SeriesInstanceUID",
            "Anatomical_Plane"
        ] + TARGETS
    ].to_string(index=False)
)

In [ ]:
# ==========================================
# CHECK WHETHER LABELS ARE IDENTICAL
# ACROSS PLANES WITHIN EACH STUDY
# ==========================================

def check_study_label_consistency(df, targets):
    inconsistent_studies = []

    for study_id, group in df.groupby("StudyInstanceUID"):

        # Remove duplicate label rows if necessary
        label_values = group[targets].drop_duplicates()

        # More than one unique label combination?
        if len(label_values) > 1:
            inconsistent_studies.append(study_id)

    return inconsistent_studies


inconsistent_studies = check_study_label_consistency(
    merged_df,
    TARGETS
)

print("\n" + "="*60)
print("LABEL CONSISTENCY CHECK")
print("="*60)

print("Total studies:",
      merged_df["StudyInstanceUID"].nunique())

print("Studies with different labels across series:",
      len(inconsistent_studies))

print(
    "Percentage:",
    round(
        100 * len(inconsistent_studies) /
        merged_df["StudyInstanceUID"].nunique(),
        2
    ),
    "%"
)

In [ ]:
# ==========================================
# PLANES AVAILABLE PER STUDY
# ==========================================

planes_per_study = (
    merged_df
    .groupby("StudyInstanceUID")["Anatomical_Plane"]
    .apply(lambda x: tuple(sorted(x.unique())))
)

print("\n" + "="*60)
print("PLANE COMBINATIONS PER STUDY")
print("="*60)

print(
    planes_per_study.value_counts()
)

In [ ]:
# ==========================================
# DISPLAY CURRENT MASKS
# ==========================================

mask_df = pd.DataFrame(
    MASKS,
    index=TARGETS
)

print("\n" + "="*60)
print("CURRENT MASKS")
print("="*60)

print(mask_df)

In [ ]:
# ==========================================
# NUMBER OF ACTIVE TARGETS PER PLANE
# ==========================================

for plane, mask in MASKS.items():
    active_targets = [
        target
        for target, value in zip(TARGETS, mask)
        if value == 1
    ]

    ignored_targets = [
        target
        for target, value in zip(TARGETS, mask)
        if value == 0
    ]

    print("\n" + "="*60)
    print(plane.upper())
    print("="*60)

    print("Training on:")
    print(active_targets)

    print("\nIgnoring:")
    print(ignored_targets)

In [ ]:
# ==========================================
# MASK + DATA COMPARISON
# ==========================================

rate_table = (
    merged_df
    .groupby("Anatomical_Plane")[TARGETS]
    .mean()
    .T
)

comparison = rate_table.copy()

for plane in ["Sagittal", "Coronal", "Axial"]:
    if plane in comparison.columns:
        comparison[plane + "_mask"] = [
            MASKS[plane][i]
            for i in range(len(TARGETS))
        ]

print("\n" + "="*60)
print("DATA RATE + CURRENT MASK")
print("="*60)

comparison.round(4)

In [ ]:
import matplotlib.pyplot as plt

# ==========================================
# HEATMAP OF POSITIVE RATE
# ==========================================

plt.figure(figsize=(10, 8))

plt.imshow(
    plane_comparison.values,
    aspect="auto"
)

plt.xticks(
    range(len(plane_comparison.columns)),
    plane_comparison.columns
)

plt.yticks(
    range(len(plane_comparison.index)),
    plane_comparison.index
)

plt.xlabel("Anatomical Plane")
plt.ylabel("Abnormality")
plt.title("Positive Label Rate (%) by Anatomical Plane")

plt.colorbar(label="Positive Rate (%)")

plt.tight_layout()
plt.show()

In [ ]:


# # ==========================================
# # 1. Configuration & Hyperparameters
# # ==========================================
# DATA_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
# LABELS_PATH = Path("/kaggle/input/datasets/yuvrajbhatkariya/data-2/new_labeled.csv") 
# DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# TARGETS = [
#     "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", 
#     "Medial OA", "Lateral OA", "PF OA", "Effusion", 
#     "Synovitis", "Baker's", "Contusion", "Fracture"
# ]

# # We recalculated these earlier to handle extreme class imbalance
# CLASS_WEIGHTS = torch.tensor([
#     4.06, 2.67, 1.31, 2.37, 5.41, 11.63, 
#     3.52, 1.79, 9.96, 8.66, 16.15, 28.98
# ], dtype=torch.float32).to(DEVICE)

# # View-Specific Masks (1 = Train, 0 = Ignore)
# MASKS = {
#     'Sagittal': [1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1],
#     'Coronal':  [0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1],
#     'Axial':    [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1]
# }

# # ==========================================
# # 2. Data Preparation (Merging Series & Labels)
# # ==========================================
# print("Loading and preparing data...")
# labels_df = pd.read_csv(LABELS_PATH)
# series_df = pd.read_csv(DATA_DIR / "train_series.csv")

# # Merge so we know the plane and the labels for every series
# merged_df = pd.merge(series_df, labels_df, on="StudyInstanceUID", how="inner")

# # Patient-Level Train/Val Split (Ensures no patient leaks into validation)
# gkf = GroupKFold(n_splits=5)
# train_idx, val_idx = next(gkf.split(merged_df, groups=merged_df['StudyInstanceUID']))

# train_data = merged_df.iloc[train_idx].reset_index(drop=True)
# val_data = merged_df.iloc[val_idx].reset_index(drop=True)
# print(f"Train Series: {len(train_data)} | Val Series: {len(val_data)}")

# # ==========================================
# # 3. Medical Image Processing & Dataset
# # ==========================================
# def apply_mri_normalization(dicom_obj):
#     img = dicom_obj.pixel_array.astype(np.float32)
#     p1, p99 = np.percentile(img, 1), np.percentile(img, 99)
#     img = np.clip(img, p1, p99)
#     if p99 > p1: img = (img - p1) / (p99 - p1)
#     else: img = img - p1
#     return (img * 255.0).astype(np.uint8)

# train_transforms = A.Compose([
#     A.Resize(256, 256),
#     A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
#     A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.5),
#     A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
#     ToTensorV2()
# ])

# val_transforms = A.Compose([
#     A.Resize(256, 256),
#     A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
#     ToTensorV2()
# ])

# class RSNAMultiChunkDataset(Dataset):
#     def __init__(self, df, data_dir, transforms=None):
#         self.data = df
#         self.data_dir = data_dir
#         self.transforms = transforms

#     def __len__(self): return len(self.data)

#     def _get_chunk(self, dcm_paths, center_idx):
#         center_idx = max(1, min(center_idx, len(dcm_paths) - 2))
#         slice_indices = [center_idx - 1, center_idx, center_idx + 1]
        
#         channels = []
#         for i in slice_indices:
#             try: channels.append(apply_mri_normalization(pydicom.dcmread(dcm_paths[i])))
#             except: channels.append(None)
                
#         valid_imgs = [img for img in channels if img is not None]
#         if not valid_imgs: return np.zeros((256, 256, 3), dtype=np.uint8)
            
#         channels = [img if img is not None else valid_imgs[0] for img in channels]
#         base_shape = channels[0].shape
#         for i in range(1, 3):
#             if channels[i].shape != base_shape:
#                 channels[i] = cv2.resize(channels[i], (base_shape[1], base_shape[0]))
#         return np.stack(channels, axis=-1)

#     def __getitem__(self, idx):
#         row = self.data.iloc[idx]
#         series_path = self.data_dir / "train_series" / str(row['StudyInstanceUID']) / str(row['SeriesInstanceUID'])
        
#         dcm_paths = sorted(list(series_path.glob("*.dcm")), key=lambda x: int(x.stem) if x.stem.isdigit() else str(x))
#         n_slices = len(dcm_paths)
        
#         if n_slices >= 3:
#             anchors = [int(n_slices * 0.25), int(n_slices * 0.50), int(n_slices * 0.75)]
#             chunks = [self._get_chunk(dcm_paths, a) for a in anchors]
#         else:
#             single = np.zeros((256, 256, 3), dtype=np.uint8)
#             chunks = [single, single, single]
            
#         processed = [self.transforms(image=c)["image"] for c in chunks] if self.transforms else chunks
#         labels = torch.tensor(row[TARGETS].values.astype(np.float32))
#         return torch.stack(processed, dim=0), labels

# # ==========================================
# # 4. Architecture & Smoothed Loss
# # ==========================================
# class RSNAFeatureAveragingModel(nn.Module):
#     def __init__(self, backbone='resnet34'):
#         super().__init__()
#         self.encoder = timm.create_model(backbone, pretrained=True, num_classes=0)
#         self.head = nn.Linear(self.encoder.num_features, 12)

#     def forward(self, x):
#         b, n, c, h, w = x.shape
#         x = x.view(b * n, c, h, w)
#         feat = self.encoder(x).view(b, n, -1).mean(dim=1)
#         return self.head(feat)

# class SmoothedMaskedWeightedBCE(nn.Module):
#     def __init__(self, pos_weights, view_mask, device, smoothing=0.15):
#         super().__init__()
#         self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weights, reduction='none')
#         self.view_mask = torch.tensor(view_mask, dtype=torch.float32).to(device)
#         self.smoothing = smoothing

#     def forward(self, logits, targets):
#         # 1.0 -> 0.925 | 0.0 -> 0.075
#         smoothed = targets * (1.0 - self.smoothing) + (self.smoothing / 2.0)
#         loss = self.bce(logits, smoothed) * self.view_mask
#         valid = self.view_mask.sum() * logits.size(0)
#         return loss.sum() / valid if valid > 0 else torch.tensor(0.0, requires_grad=True).to(logits.device)

# # ==========================================
# # 5. Training Engine
# # ==========================================
# def train_specialist_model(plane_name, mask):
#     print(f"\n{'='*40}\n🚀 TRAINING {plane_name.upper()} SPECIALIST\n{'='*40}")
    
#     # Filter Data
#     t_df = train_data[train_data['Anatomical_Plane'] == plane_name].reset_index(drop=True)
#     v_df = val_data[val_data['Anatomical_Plane'] == plane_name].reset_index(drop=True)
    
#     train_loader = DataLoader(RSNAMultiChunkDataset(t_df, DATA_DIR, train_transforms), batch_size=8, shuffle=True, num_workers=2, persistent_workers=True)
#     val_loader = DataLoader(RSNAMultiChunkDataset(v_df, DATA_DIR, val_transforms), batch_size=8, shuffle=False, num_workers=2, persistent_workers=True)
    
#     # Initialize
#     model = RSNAFeatureAveragingModel(backbone='resnet34').to(DEVICE)
#     optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
#     criterion = SmoothedMaskedWeightedBCE(pos_weights=CLASS_WEIGHTS, view_mask=mask, device=DEVICE, smoothing=0.15)
    
#     best_auc = 0.0
#     EPOCHS = 8 # Sweet spot for ResNet34 with this dataset size
    
#     for epoch in range(EPOCHS):
#         model.train()
#         running_loss = 0.0
        
#         for imgs, lbls in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
#             imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
#             optimizer.zero_grad()
#             loss = criterion(model(imgs), lbls)
#             loss.backward()
#             optimizer.step()
#             running_loss += loss.item()
            
#         model.eval()
#         preds, targets = [], []
#         with torch.no_grad():
#             for imgs, lbls in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"):
#                 preds.append(torch.sigmoid(model(imgs.to(DEVICE))).cpu().numpy())
#                 targets.append(lbls.numpy())
                
#         preds, targets = np.vstack(preds), np.vstack(targets)
        
#         aucs = []
#         for i in range(12):
#             if mask[i] == 1:
#                 try: aucs.append(roc_auc_score(targets[:, i], preds[:, i]))
#                 except: pass
                
#         macro_auc = np.mean(aucs) if aucs else 0
#         print(f"Loss: {running_loss/len(train_loader):.4f} | Valid Macro-AUC: {macro_auc:.4f}")
        
#         if macro_auc > best_auc:
#             best_auc = macro_auc
#             torch.save(model.state_dict(), f"/kaggle/working/best_{plane_name.lower()}_model.pth")
#             print(f"🏆 New Best {plane_name} Model Saved!")

#     # Cleanup memory before the next model starts
#     del model, optimizer, train_loader, val_loader
#     torch.cuda.empty_cache()

# # ==========================================
# # 6. Execution Queue
# # ==========================================
# # Train all three sequentially
# train_specialist_model('Sagittal', MASKS['Sagittal'])
# train_specialist_model('Coronal', MASKS['Coronal'])
# train_specialist_model('Axial', MASKS['Axial'])

# print("\n🎉 ALL SPECIALIST MODELS TRAINED SUCCESSFULLY!")